## Stemming

`Stemming` is the process of reducing a word to its word stem by removing affixes (prefixes or suffixes).

Why do we need it?
- Example task: review classification (positive or negative).
-Different forms carry the same meaning for this task: eating, eat, eaten share the stem eat; going, gone, goes share go.
- Keeping every variant increases the number of input features. Reducing to one stem shrinks the vocabulary and keeps the model simpler.

In [2]:
## words ---> eating, eat, eaten ---> root word ---> eat 

words=["eating","eats","eaten","writing","writes","programming","programs","history","finally","finalized"]

#### 1. PorterStemmer

In [13]:
from nltk.stem import PorterStemmer

porterstemming = PorterStemmer()

In [14]:
for word in words:
    print(word + "--->" + porterstemming.stem(word))

eating--->eat
eats--->eat
eaten--->eaten
writing--->write
writes--->write
programming--->program
programs--->program
history--->histori
finally--->final
finalized--->final


In [16]:
porterstemming.stem('congratulations')

'congratul'

In [17]:
porterstemming.stem("sitting")

'sit'

- history becomes histori, changing the form and meaning.
- congratulations becomes congratul, which is not a real word.
- sitting becomes sit, which is good.
- `Stemming` works for many words but not all.

#### 2. RegexpStemmer class

In [26]:
from nltk.stem import RegexpStemmer

reg_stemmer = RegexpStemmer('ing$|s$|e$|able$', min=4)

- Takes a regular expression and removes any prefix or suffix that matches.
- `$` means "at the end of the word". Without it, the pattern matches anywhere and every match is removed.
- `min=4` is the minimum word length required before stemming is applied.

In [9]:
reg_stemmer.stem('eating')

'eat'

In [10]:
reg_stemmer.stem('ingeating')

'ingeat'

In [31]:
def show(pattern, words, min=0):
    s = RegexpStemmer(pattern, min=min)
    print(pattern, "->", [s.stem(w) for w in words])

# CASE 1: your pattern, fixed with $ on every suffix
show('ing$|s$|e$|able$', ["eating", "eats", "hope", "controlable", "reading", "sing"], min=4)
# ['eat', 'eat', 'hop', 'control', 'read', 's']
# note: "sing" is over-stemmed to "s" (see case 5)

# CASE 2: with $ vs without $
show('ing',  ["eating", "ingeating", "singing"])
# ['eat', 'eat', 's']          every "ing" removed, anywhere
show('ing$', ["eating", "ingeating", "singing"])
# ['eat', 'ingeat', 'sing']    only the final "ing" removed

# CASE 3: prefixes with ^ (and what goes wrong without it)
show('^un|^re', ["unhappy", "redo", "rerun", "sun", "unreal"])
# ['happy', 'do', 'run', 'sun', 'real']      only the start is touched
show('un|re',   ["unhappy", "redo", "rerun", "sun", "unreal"])
# ['happy', 'do', 'r', 's', 'al']            matches anywhere, mangles words

# CASE 4: min length protects short words
show('s$', ["is", "as", "bus", "cats", "goes"], min=0)
# ['i', 'a', 'bu', 'cat', 'goe']
show('s$', ["is", "as", "bus", "cats", "goes"], min=4)
# ['is', 'as', 'bus', 'cats' -> 'cat', 'goes' -> 'goe']  (is, as, bus untouched)

# CASE 5: min=4 vs min=5 (len(word) < min is skipped)
show('ing$', ["sing", "king", "eating"], min=4)
# ['s', 'k', 'eat']            4-letter words are still stemmed
show('ing$', ["sing", "king", "eating"], min=5)
# ['sing', 'king', 'eat']      4-letter words are protected

# CASE 6: no match means the word is unchanged
show('ing$|ly$', ["book", "tree", "quickly"], min=4)
# ['book', 'tree', 'quick']

# CASE 7: order of alternatives at the same starting point
show('in|ing', ["singing"])
# ['sgg']                      "in" matched first, leaving the "g"
show('ing|in', ["singing"])
# ['s']                        "ing" matched first, the full suffix goes

# CASE 8: case sensitivity and the (?i) flag
show('ing$',    ["EATING", "Eating", "eating"])
# ['EATING', 'Eat', 'eat']     uppercase suffix not matched
show('(?i)ing$', ["EATING", "Eating", "eating"])
# ['EAT', 'Eat', 'eat']        (?i) makes it case-insensitive

# CASE 9: several suffixes in one rule
show('ed$|ing$|ly$|ness$', ["played", "playing", "quickly", "kindness", "play"], min=4)
# ['play', 'play', 'quick', 'kind', 'play']

# CASE 10: regex classes for cleaning, not only suffixes
show('[0-9]+', ["abc123", "2024year", "a1b2"])
# ['abc', 'year', 'ab']        removes digits
show('[^a-z]', ["Hello!", "can't", "x-ray"])
# ['ello', 'cant', 'xray']     removes anything that is not a lowercase letter

# CASE 11: over-stemming and non-words (no dictionary)
show('ies$', ["babies", "studies"])
# ['bab', 'stud']              not real words
show('ies$|y$', ["babies", "happy"])
# ['bab', 'happ']

ing$|s$|e$|able$ -> ['eat', 'eat', 'hop', 'control', 'read', 's']
ing -> ['eat', 'eat', 's']
ing$ -> ['eat', 'ingeat', 'sing']
^un|^re -> ['happy', 'do', 'run', 'sun', 'real']
un|re -> ['happy', 'do', 'r', 's', 'al']
s$ -> ['i', 'a', 'bu', 'cat', 'goe']
s$ -> ['is', 'as', 'bus', 'cat', 'goe']
ing$ -> ['s', 'k', 'eat']
ing$ -> ['sing', 'king', 'eat']
ing$|ly$ -> ['book', 'tree', 'quick']
in|ing -> ['sgg']
ing|in -> ['s']
ing$ -> ['EATING', 'Eat', 'eat']
(?i)ing$ -> ['EAT', 'Eat', 'eat']
ed$|ing$|ly$|ness$ -> ['play', 'play', 'quick', 'kind', 'play']
[0-9]+ -> ['abc', 'year', 'ab']
[^a-z] -> ['ello', 'cant', 'xray']
ies$ -> ['bab', 'stud']
ies$|y$ -> ['bab', 'happ']


#### 3. Snowball Stemmer

- Performs better than Porter and supports multiple languages (Arabic, Danish, English, Finnish, French, German, Hungarian, Italian and more).

In [32]:
from nltk.stem import SnowballStemmer

snowballstemmer = SnowballStemmer('english')

In [33]:
for word in words:
    print(word + "--->" + snowballstemmer.stem(word))

eating--->eat
eats--->eat
eaten--->eaten
writing--->write
writes--->write
programming--->program
programs--->program
history--->histori
finally--->final
finalized--->final


In [34]:
porterstemming.stem('fairly'), porterstemming.stem("sportingly")

('fairli', 'sportingli')

In [35]:
snowballstemmer.stem('fairly'), snowballstemmer.stem("sportingly")

('fair', 'sport')

In [36]:
snowballstemmer.stem('goes')

'goe'

In [37]:
stemming.stem('goes')

'goe'